# Research: Sector Momentum ETF Rotation (Issue #22)

## Contexte
- **Performance actuelle**: Sharpe 0.216, CAGR 6.5%, MaxDD 29.9%
- **Problemes**: MaxDD 30% (TLT risk-off en 2022), poids arbitraires, top 3 concentre
- **Objectif**: Sharpe > 0.4, MaxDD < 20%

## Hypotheses a tester
1. Nombre optimal de positions (2-5)
2. Lookback scheme : 1 mois seul vs composite vs 6 mois seul
3. TLT risk-off : cash > TLT en 2022 (taux montants)
4. Volatility-adjusted momentum (diviser par vol sectorielle)
5. Filtre de momentum absolu (ne trader que si momentum > 0)

## References
- Mebane Faber (2007) "Quantitative Approach to Tactical Asset Allocation"
- Jegadeesh & Titman (1993) "Returns to Buying Winners"

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 11 GICS sector ETFs + SPY + TLT
sector_tickers = ['XLK', 'XLF', 'XLE', 'XLV', 'XLI', 'XLY', 'XLP', 'XLU', 'XLB', 'XLRE', 'XLC']
all_tickers = sector_tickers + ['SPY', 'TLT']

data = yf.download(all_tickers, start='2010-01-01', end='2026-01-01')
closes = data['Close'].dropna()

# Rendements
returns = closes.pct_change()

print(f"Periode: {closes.index[0].date()} a {closes.index[-1].date()}")
print(f"\nRendements annualises (buy & hold):")
for t in sector_tickers:
    ret = (closes[t].iloc[-1] / closes[t].iloc[0]) ** (252/len(closes)) - 1
    vol = returns[t].std() * np.sqrt(252)
    print(f"  {t}: {ret:.1%} (vol: {vol:.1%})")

[                       0%                       ]

[***********           23%                       ]  3 of 13 completed

[***************       31%                       ]  4 of 13 completed

[***************       31%                       ]  4 of 13 completed

[**********************46%                       ]  6 of 13 completed

...

[**********************77%************           ]  10 of 13 completed

[**********************85%****************       ]  11 of 13 completed

[**********************92%*******************    ]  12 of 13 completed

[*********************100%***********************]  13 of 13 completed

Periode: 2018-06-19 a 2025-12-31

Rendements annualises (buy & hold):
  XLK: 21.5% (vol: 26.6%)
  XLF: 11.8% (vol: 23.8%)
  XLE: 7.1% (vol: 32.3%)
  XLV: 10.2% (vol: 17.5%)
  XLI: 12.4% (vol: 21.5%)
  XLY: 11.7% (vol: 24.2%)
  XLP: 8.6% (vol: 15.6%)
  XLU: 10.7% (vol: 20.7%)
  XLB: 8.2% (vol: 22.2%)
  XLRE: 6.9% (vol: 22.3%)
  XLC: 13.2% (vol: 22.6%)


## Hypothese 1: Nombre de positions et lookback

Grid search: nombre de positions x type de momentum

In [2]:
def sector_momentum_backtest(closes, sector_tickers, spy_col='SPY', tlt_col='TLT',
                              top_n=3, lookback=126, rebal_freq=21,
                              use_risk_off=True, risk_off_asset='TLT',
                              abs_momentum_filter=False, vol_adjust=False):
    """Backtest sector momentum rotation."""
    returns_df = closes.pct_change()
    sma200 = closes[spy_col].rolling(200).mean()
    
    portfolio_returns = []
    holdings = []
    rebal_counter = rebal_freq
    
    start_idx = max(lookback, 200) + 1
    
    for i in range(start_idx, len(closes)):
        # Daily return from current holdings
        if holdings:
            port_ret = np.mean([returns_df[t].iloc[i] for t in holdings])
        else:
            port_ret = 0
        portfolio_returns.append(port_ret)
        
        rebal_counter += 1
        if rebal_counter < rebal_freq:
            continue
        rebal_counter = 0
        
        # SMA200 regime filter
        spy_price = closes[spy_col].iloc[i]
        spy_sma = sma200.iloc[i]
        if pd.isna(spy_sma):
            continue
        
        risk_on = spy_price > spy_sma
        
        if not risk_on:
            if use_risk_off and risk_off_asset:
                holdings = [risk_off_asset]
            else:
                holdings = []  # Cash
            continue
        
        # Calculate momentum scores
        scores = {}
        for t in sector_tickers:
            current = closes[t].iloc[i]
            past = closes[t].iloc[i - lookback]
            if past > 0:
                mom = current / past - 1
                if vol_adjust:
                    vol = returns_df[t].iloc[i-63:i].std() * np.sqrt(252)
                    if vol > 0:
                        mom = mom / vol
                scores[t] = mom
        
        if len(scores) < top_n:
            continue
        
        sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        
        if abs_momentum_filter:
            top = [t for t, s in sorted_scores[:top_n] if s > 0]
        else:
            top = [t for t, s in sorted_scores[:top_n]]
        
        if len(top) == 0:
            if use_risk_off and risk_off_asset:
                holdings = [risk_off_asset]
            else:
                holdings = []
        else:
            holdings = top
    
    rets = np.array(portfolio_returns)
    cum = pd.Series((1 + rets).cumprod())
    total = cum.iloc[-1] - 1
    years = len(rets) / 252
    cagr = (1 + total) ** (1/years) - 1
    vol = rets.std() * np.sqrt(252)
    sharpe = (cagr - 0.04) / vol if vol > 0 else 0
    max_dd = ((cum / cum.cummax()) - 1).min()
    
    return {'sharpe': sharpe, 'cagr': cagr, 'max_dd': max_dd, 'vol': vol, 'cum': cum}

# Grid search: top_n x lookback
print(f"{'Top_N':<7} {'Lookback':<10} {'Sharpe':>7} {'CAGR':>7} {'MaxDD':>7} {'Vol':>7}")
print("-" * 50)

best = {'sharpe': -999}
for top_n in [2, 3, 4, 5]:
    for lb in [21, 63, 126, 252]:
        r = sector_momentum_backtest(closes, sector_tickers, top_n=top_n, lookback=lb)
        print(f"{top_n:<7} {lb:<10} {r['sharpe']:>7.3f} {r['cagr']:>6.1%} {r['max_dd']:>6.1%} {r['vol']:>6.1%}")
        if r['sharpe'] > best['sharpe']:
            best = r
            best['config'] = (top_n, lb)

print(f"\nBest: top_n={best['config'][0]}, lookback={best['config'][1]}, Sharpe={best['sharpe']:.3f}")

Top_N   Lookback    Sharpe    CAGR   MaxDD     Vol
--------------------------------------------------
2       21          -0.340  -2.2% -40.2%  18.4%
2       63           0.042   4.8% -31.3%  18.8%
2       126          0.107   6.0% -35.5%  19.1%
2       252          0.153   7.0% -36.8%  19.8%
3       21          -0.198   0.5% -38.4%  17.4%
3       63          -0.088   2.4% -29.0%  17.9%
3       126          0.063   5.1% -32.3%  18.0%


3       252          0.137   6.6% -35.7%  18.7%
4       21          -0.122   1.9% -35.0%  17.0%
4       63          -0.129   1.8% -32.5%  17.2%
4       126          0.042   4.7% -30.5%  17.2%
4       252         -0.002   4.0% -37.4%  18.2%
5       21          -0.117   2.1% -31.8%  16.6%
5       63          -0.080   2.6% -32.4%  16.8%
5       126          0.048   4.8% -30.6%  16.7%


5       252         -0.021   3.6% -35.6%  17.6%

Best: top_n=2, lookback=252, Sharpe=0.153


## Hypothese 2: TLT risk-off vs cash

TLT a perdu ~30% en 2022 (hausse des taux). Cash est plus sur.

In [3]:
tn, lb = best['config']
print(f"Config: top_{tn}, lookback={lb}")
print()

# Compare TLT vs Cash risk-off
r_tlt = sector_momentum_backtest(closes, sector_tickers, top_n=tn, lookback=lb, use_risk_off=True, risk_off_asset='TLT')
r_cash = sector_momentum_backtest(closes, sector_tickers, top_n=tn, lookback=lb, use_risk_off=True, risk_off_asset=None)
r_xlp = sector_momentum_backtest(closes, sector_tickers, top_n=tn, lookback=lb, use_risk_off=True, risk_off_asset='XLP')

print(f"{'Risk-off':<15} {'Sharpe':>7} {'CAGR':>7} {'MaxDD':>7}")
print("-" * 40)
print(f"{'TLT':<15} {r_tlt['sharpe']:>7.3f} {r_tlt['cagr']:>6.1%} {r_tlt['max_dd']:>6.1%}")
print(f"{'Cash':<15} {r_cash['sharpe']:>7.3f} {r_cash['cagr']:>6.1%} {r_cash['max_dd']:>6.1%}")
print(f"{'XLP (staples)':<15} {r_xlp['sharpe']:>7.3f} {r_xlp['cagr']:>6.1%} {r_xlp['max_dd']:>6.1%}")

Config: top_2, lookback=252

Risk-off         Sharpe    CAGR   MaxDD
----------------------------------------
TLT               0.153   7.0% -36.8%
Cash              0.393  11.0% -35.7%
XLP (staples)     0.450  13.0% -33.1%


## Hypothese 3: Volatility-adjusted momentum

Diviser le momentum par la volatilite sectorielle pour favoriser
les trends "propres" (fort momentum + faible vol).

In [4]:
tn, lb = best['config']

r_raw = sector_momentum_backtest(closes, sector_tickers, top_n=tn, lookback=lb, vol_adjust=False)
r_vol = sector_momentum_backtest(closes, sector_tickers, top_n=tn, lookback=lb, vol_adjust=True)
r_abs = sector_momentum_backtest(closes, sector_tickers, top_n=tn, lookback=lb, abs_momentum_filter=True)
r_both = sector_momentum_backtest(closes, sector_tickers, top_n=tn, lookback=lb, vol_adjust=True, abs_momentum_filter=True)

print(f"Config: top_{tn}, lookback={lb}")
print(f"{'Mode':<25} {'Sharpe':>7} {'CAGR':>7} {'MaxDD':>7}")
print("-" * 50)
print(f"{'Raw momentum':<25} {r_raw['sharpe']:>7.3f} {r_raw['cagr']:>6.1%} {r_raw['max_dd']:>6.1%}")
print(f"{'Vol-adjusted':<25} {r_vol['sharpe']:>7.3f} {r_vol['cagr']:>6.1%} {r_vol['max_dd']:>6.1%}")
print(f"{'Abs momentum filter':<25} {r_abs['sharpe']:>7.3f} {r_abs['cagr']:>6.1%} {r_abs['max_dd']:>6.1%}")
print(f"{'Vol-adj + abs filter':<25} {r_both['sharpe']:>7.3f} {r_both['cagr']:>6.1%} {r_both['max_dd']:>6.1%}")

Config: top_2, lookback=252

Mode                       Sharpe    CAGR   MaxDD
--------------------------------------------------
Raw momentum                0.153   7.0% -36.8%
Vol-adjusted               -0.013   3.7% -36.5%
Abs momentum filter         0.167   7.3% -36.8%
Vol-adj + abs filter        0.002   4.0% -36.5%


## Hypothese 4: Composite vs simple momentum

La strategie actuelle utilise un composite (1, 3, 6, 12 mois).
Testons si un lookback simple est meilleur.

In [5]:
def composite_momentum_backtest(closes, sector_tickers, top_n=3,
                                  lookbacks=[21, 63, 126, 252],
                                  weights=[0.4, 0.2, 0.2, 0.2],
                                  rebal_freq=21, risk_off_asset=None):
    """Backtest with composite momentum."""
    returns_df = closes.pct_change()
    sma200 = closes['SPY'].rolling(200).mean()
    max_lb = max(lookbacks)
    
    portfolio_returns = []
    holdings = []
    rebal_counter = rebal_freq
    
    for i in range(max_lb + 201, len(closes)):
        if holdings:
            port_ret = np.mean([returns_df[t].iloc[i] for t in holdings])
        else:
            port_ret = 0
        portfolio_returns.append(port_ret)
        
        rebal_counter += 1
        if rebal_counter < rebal_freq:
            continue
        rebal_counter = 0
        
        if closes['SPY'].iloc[i] <= sma200.iloc[i]:
            holdings = [risk_off_asset] if risk_off_asset else []
            continue
        
        scores = {}
        for t in sector_tickers:
            score = 0
            valid = True
            for lb, w in zip(lookbacks, weights):
                past = closes[t].iloc[i - lb]
                if past <= 0:
                    valid = False
                    break
                score += w * (closes[t].iloc[i] / past - 1)
            if valid:
                scores[t] = score
        
        sorted_s = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        top = [t for t, s in sorted_s[:top_n] if s > 0]
        holdings = top if top else ([risk_off_asset] if risk_off_asset else [])
    
    rets = np.array(portfolio_returns)
    cum = pd.Series((1 + rets).cumprod())
    total = cum.iloc[-1] - 1
    years = len(rets) / 252
    cagr = (1 + total) ** (1/years) - 1
    vol = rets.std() * np.sqrt(252)
    sharpe = (cagr - 0.04) / vol if vol > 0 else 0
    max_dd = ((cum / cum.cummax()) - 1).min()
    
    return {'sharpe': sharpe, 'cagr': cagr, 'max_dd': max_dd}

# Test different composite schemes
schemes = [
    ('1m only', [21], [1.0]),
    ('3m only', [63], [1.0]),
    ('6m only', [126], [1.0]),
    ('12m only', [252], [1.0]),
    ('Current 4/2/2/2', [21, 63, 126, 252], [0.4, 0.2, 0.2, 0.2]),
    ('Equal 1/3/6/12', [21, 63, 126, 252], [0.25, 0.25, 0.25, 0.25]),
    ('Recent 6/3/1', [21, 63, 126], [0.5, 0.3, 0.2]),
    ('Short 1/3', [21, 63], [0.6, 0.4]),
]

print(f"{'Scheme':<20} {'Sharpe':>7} {'CAGR':>7} {'MaxDD':>7}")
print("-" * 45)

for name, lbs, wts in schemes:
    r = composite_momentum_backtest(closes, sector_tickers, top_n=3, lookbacks=lbs, weights=wts)
    print(f"{name:<20} {r['sharpe']:>7.3f} {r['cagr']:>6.1%} {r['max_dd']:>6.1%}")

Scheme                Sharpe    CAGR   MaxDD
---------------------------------------------
1m only                0.032   4.5% -29.3%
3m only                0.181   6.6% -22.2%
6m only                0.337   9.0% -23.8%
12m only               0.612  12.8% -14.0%
Current 4/2/2/2        0.490  11.0% -16.7%
Equal 1/3/6/12         0.451  10.5% -18.1%
Recent 6/3/1           0.284   8.2% -25.7%
Short 1/3              0.205   7.0% -28.1%


## Conclusions et recommandations

### Configuration recommandee
Basee sur les resultats ci-dessus, identifier:
1. Nombre optimal de positions
2. Type de momentum (simple vs composite)
3. Risk-off: cash vs TLT
4. Filtres additionnels (vol-adjust, abs momentum)

## Hypothese 5: Skip-Month (Jegadeesh 1990)

**Question**: Exclure le dernier mois du lookback ameliore-t-il les resultats ?

La v3.0 QC utilise `12m-1m` (lookback=252, skip=21). Le skip-month evite la contamination
par la mean-reversion a court terme. Testons son impact sur la config optimale identifiee
jusqu'ici (12m only, top_n=3, risk-off=XLP/cash).

Note: le backtester existant utilise un rebalancement 21j. Ici on ajoute le skip-month
en comparant lookback=252 standard vs 252 avec skip final de 21j.

In [ ]:
def backtest_with_skip(closes, sector_tickers, top_n=3, lookback=252, skip_days=0,
                        vol_window=63, vol_adjust=False, abs_filter=True,
                        risk_off_asset=None, rebal_freq=21):
    """
    Backtester avec support du skip-month.
    skip_days=0  -> momentum = price[t] / price[t-lookback] - 1
    skip_days=21 -> momentum = price[t-21] / price[t-lookback] - 1 (12m-1m)
    """
    returns_df = closes.pct_change()
    sma200 = closes['SPY'].rolling(200).mean()
    sma20  = closes['SPY'].rolling(20).mean()

    portfolio_returns = []
    holdings = []
    rebal_counter = rebal_freq

    start_idx = max(lookback + skip_days, 200) + 1

    for i in range(start_idx, len(closes)):
        if holdings:
            port_ret = np.mean([returns_df[t].iloc[i] for t in holdings if t in closes.columns])
        else:
            port_ret = 0
        portfolio_returns.append(port_ret)

        rebal_counter += 1
        if rebal_counter < rebal_freq:
            continue
        rebal_counter = 0

        spy_price = closes['SPY'].iloc[i]
        spy_sma200 = sma200.iloc[i]
        spy_sma20  = sma20.iloc[i]
        if pd.isna(spy_sma200) or pd.isna(spy_sma20):
            continue

        # Risk-off: SPY < SMA200 AND SPY < SMA20 (meme logique que v3.0)
        risk_off = (spy_price < spy_sma200) and (spy_price < spy_sma20)

        if risk_off:
            holdings = [risk_off_asset] if risk_off_asset else []
            continue

        # Calcul scores
        scores = {}
        raw_scores = {}
        for t in sector_tickers:
            if i - lookback - skip_days < 0:
                continue
            price_current = closes[t].iloc[i - skip_days] if skip_days > 0 else closes[t].iloc[i]
            price_past    = closes[t].iloc[i - lookback]
            if price_past <= 0 or price_current <= 0:
                continue
            raw_mom = price_current / price_past - 1
            raw_scores[t] = raw_mom

            if vol_adjust:
                vol_slice = returns_df[t].iloc[i - vol_window:i]
                vol = vol_slice.std()
                score = raw_mom / vol if vol > 0 else 0
            else:
                score = raw_mom
            scores[t] = score

        if len(scores) < top_n:
            continue

        sorted_s = sorted(scores.items(), key=lambda x: x[1], reverse=True)

        if abs_filter:
            top = [t for t, s in sorted_s[:top_n] if raw_scores.get(t, 0) > 0]
        else:
            top = [t for t, s in sorted_s[:top_n]]

        if not top:
            holdings = [risk_off_asset] if risk_off_asset else []
        else:
            holdings = top

    rets = np.array(portfolio_returns)
    cum = pd.Series((1 + rets).cumprod())
    total = cum.iloc[-1] - 1
    years = len(rets) / 252
    cagr = (1 + total) ** (1/years) - 1
    vol_ann = rets.std() * np.sqrt(252)
    sharpe = (cagr - 0.025) / vol_ann if vol_ann > 0 else 0  # rf=2.5% (plus realiste 2018-2026)
    max_dd = ((cum / cum.cummax()) - 1).min()

    return {'sharpe': sharpe, 'cagr': cagr, 'max_dd': max_dd, 'vol': vol_ann, 'cum': cum}


# Test skip-month sur config optimale: 12m only, top_n=3, abs_filter, risk_off=cash
# Note: rf=2.5% ici (les resultats precedents utilisaient rf=4%, valeur 2010-era)
print("=== H5: Skip-Month vs No-Skip (config: 12m, top3, cash risk-off, abs filter) ===")
print(f"{'Variante':<35} {'Sharpe':>7} {'CAGR':>7} {'MaxDD':>7}")
print("-" * 60)

configs_skip = [
    ('12m no-skip (raw)',              dict(skip_days=0,  vol_adjust=False)),
    ('12m-1m skip (raw)',              dict(skip_days=21, vol_adjust=False)),
    ('12m no-skip (vol-adj)',          dict(skip_days=0,  vol_adjust=True)),
    ('12m-1m skip (vol-adj)',          dict(skip_days=21, vol_adjust=True)),
    ('12m-2w skip (raw)',              dict(skip_days=10, vol_adjust=False)),
]

for name, kwargs in configs_skip:
    r = backtest_with_skip(closes, sector_tickers, top_n=3, lookback=252,
                            abs_filter=True, risk_off_asset=None, **kwargs)
    print(f"{name:<35} {r['sharpe']:>7.3f} {r['cagr']:>6.1%} {r['max_dd']:>6.1%}")

## Hypothese 6: Stop-Loss par Position (-8% a -15%)

**Question**: Un stop-loss individuel par secteur reduit-il le MaxDD sans sacrifier le Sharpe ?

La v3.0 n'a pas de stop-loss entre deux rebalancements mensuels. Un secteur peut donc tomber
de -20% entre deux rebalancements (ex: XLE mars 2020, XLRE 2022). Un stop-loss a -10% coupe
les vrais breakdowns de tendance sans tronquer les corrections normales qui se reprennent.

In [ ]:
def backtest_with_stoploss(closes, sector_tickers, top_n=3, lookback=252, skip_days=0,
                             vol_adjust=False, abs_filter=True, risk_off_asset=None,
                             stop_loss=None, rebal_freq=21):
    """
    Backtester avec stop-loss individuel par position.
    stop_loss=None -> pas de stop
    stop_loss=-0.10 -> sortir si position perd >10% depuis l'entree
    """
    returns_df = closes.pct_change()
    sma200 = closes['SPY'].rolling(200).mean()
    sma20  = closes['SPY'].rolling(20).mean()

    portfolio_returns = []
    holdings = []
    entry_prices = {}  # {ticker: prix d'entree pour stop-loss}
    rebal_counter = rebal_freq

    start_idx = max(lookback + skip_days, 200) + 1

    for i in range(start_idx, len(closes)):
        # Verifier stop-loss quotidiennement
        if stop_loss is not None and holdings:
            stopped_out = []
            for t in holdings:
                if t in entry_prices and t in closes.columns:
                    current_p = closes[t].iloc[i]
                    entry_p = entry_prices[t]
                    if entry_p > 0 and (current_p / entry_p - 1) < stop_loss:
                        stopped_out.append(t)
            for t in stopped_out:
                holdings.remove(t)
                del entry_prices[t]

        if holdings:
            port_ret = np.mean([returns_df[t].iloc[i] for t in holdings if t in closes.columns])
        else:
            port_ret = 0
        portfolio_returns.append(port_ret)

        rebal_counter += 1
        if rebal_counter < rebal_freq:
            continue
        rebal_counter = 0

        spy_price = closes['SPY'].iloc[i]
        if pd.isna(sma200.iloc[i]) or pd.isna(sma20.iloc[i]):
            continue

        risk_off = (spy_price < sma200.iloc[i]) and (spy_price < sma20.iloc[i])

        if risk_off:
            holdings = [risk_off_asset] if risk_off_asset else []
            entry_prices = {}
            continue

        # Scores momentum
        scores = {}
        raw_scores = {}
        for t in sector_tickers:
            idx_cur = i - skip_days if skip_days > 0 else i
            idx_past = i - lookback
            if idx_past < 0:
                continue
            price_current = closes[t].iloc[idx_cur]
            price_past    = closes[t].iloc[idx_past]
            if price_past <= 0 or price_current <= 0:
                continue
            raw_mom = price_current / price_past - 1
            raw_scores[t] = raw_mom
            if vol_adjust:
                v = returns_df[t].iloc[i-63:i].std()
                score = raw_mom / v if v > 0 else 0
            else:
                score = raw_mom
            scores[t] = score

        if len(scores) < top_n:
            continue

        sorted_s = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        top = ([t for t, s in sorted_s[:top_n] if raw_scores.get(t, 0) > 0]
               if abs_filter else [t for t, s in sorted_s[:top_n]])

        if not top:
            new_holdings = [risk_off_asset] if risk_off_asset else []
        else:
            new_holdings = top

        # Mettre a jour entry_prices pour les nouveaux entrants
        for t in new_holdings:
            if t not in holdings:  # Nouvelle position
                entry_prices[t] = closes[t].iloc[i]
        # Supprimer les sortants
        for t in list(entry_prices.keys()):
            if t not in new_holdings:
                del entry_prices[t]

        holdings = new_holdings

    rets = np.array(portfolio_returns)
    cum = pd.Series((1 + rets).cumprod())
    total = cum.iloc[-1] - 1
    years = len(rets) / 252
    cagr = (1 + total) ** (1/years) - 1
    vol_ann = rets.std() * np.sqrt(252)
    sharpe = (cagr - 0.025) / vol_ann if vol_ann > 0 else 0
    max_dd = ((cum / cum.cummax()) - 1).min()

    return {'sharpe': sharpe, 'cagr': cagr, 'max_dd': max_dd, 'cum': cum}


# Benchmark: config proche de v3.0 (12m-1m, top4, vol-adj, abs-filter, cash risk-off)
print("=== H6: Impact Stop-Loss (config proche v3.0: 12m-1m, top4, vol-adj) ===")
print(f"{'Stop-Loss':<20} {'Sharpe':>7} {'CAGR':>7} {'MaxDD':>7} {'Verdict'}")
print("-" * 65)

stoploss_tests = [
    ('Aucun (baseline)',  None),
    ('SL -8%',           -0.08),
    ('SL -10%',          -0.10),
    ('SL -12%',          -0.12),
    ('SL -15%',          -0.15),
]

base_sharpe = None
for name, sl in stoploss_tests:
    r = backtest_with_stoploss(closes, sector_tickers, top_n=4, lookback=252, skip_days=21,
                                vol_adjust=True, abs_filter=True, risk_off_asset=None,
                                stop_loss=sl)
    if base_sharpe is None:
        base_sharpe = r['sharpe']
        verdict = 'baseline'
    elif r['sharpe'] > base_sharpe + 0.02:
        verdict = 'AMELIORE'
    elif r['max_dd'] > -0.28 and r['sharpe'] >= base_sharpe - 0.02:
        verdict = 'MaxDD reduit'
    else:
        verdict = 'degrade'
    print(f"{name:<20} {r['sharpe']:>7.3f} {r['cagr']:>6.1%} {r['max_dd']:>6.1%}  {verdict}")

## Hypothese 7: Double Sort (Momentum + Low Volatility)

**Question**: Un tiebreaker par volatilite basse ameliore-t-il la selection ?

Principe: parmi les secteurs ayant un momentum similaire, preferer ceux avec une volatilite
plus basse. Cela s'inspire du "low volatility anomaly" (Baker & Haugen 2012): les actifs
moins volatils surperforment sur le long terme car ils sont systematiquement sous-detenus.

Implementation: rank_combine = rank_momentum + 0.3 * rank_low_vol

In [ ]:
def backtest_double_sort(closes, sector_tickers, top_n=4, lookback=252, skip_days=21,
                          vol_window=63, abs_filter=True, risk_off_asset=None,
                          low_vol_weight=0.3, rebal_freq=21):
    """
    Double sort: rank par momentum, tiebreaker par low vol.
    low_vol_weight: poids du rank vol dans le score combine (0 = momentum pur)
    """
    returns_df = closes.pct_change()
    sma200 = closes['SPY'].rolling(200).mean()
    sma20  = closes['SPY'].rolling(20).mean()

    portfolio_returns = []
    holdings = []
    rebal_counter = rebal_freq

    start_idx = max(lookback + skip_days, 200) + 1

    for i in range(start_idx, len(closes)):
        if holdings:
            port_ret = np.mean([returns_df[t].iloc[i] for t in holdings if t in closes.columns])
        else:
            port_ret = 0
        portfolio_returns.append(port_ret)

        rebal_counter += 1
        if rebal_counter < rebal_freq:
            continue
        rebal_counter = 0

        spy_price = closes['SPY'].iloc[i]
        if pd.isna(sma200.iloc[i]) or pd.isna(sma20.iloc[i]):
            continue
        risk_off = (spy_price < sma200.iloc[i]) and (spy_price < sma20.iloc[i])
        if risk_off:
            holdings = [risk_off_asset] if risk_off_asset else []
            continue

        # Calcul scores et vols
        raw_scores = {}
        vols = {}
        for t in sector_tickers:
            idx_cur = i - skip_days if skip_days > 0 else i
            idx_past = i - lookback
            if idx_past < 0:
                continue
            pc, pp = closes[t].iloc[idx_cur], closes[t].iloc[idx_past]
            if pp <= 0 or pc <= 0:
                continue
            raw_scores[t] = pc / pp - 1
            v = returns_df[t].iloc[i - vol_window:i].std()
            vols[t] = v if v > 0 else np.nan

        valid = list(set(raw_scores.keys()) & set(vols.keys()))
        if len(valid) < top_n:
            continue

        # Appliquer filtre absolu
        if abs_filter:
            valid = [t for t in valid if raw_scores.get(t, 0) > 0]
        if not valid:
            holdings = [risk_off_asset] if risk_off_asset else []
            continue

        # Double sort: rank momentum (desc) + rank vol (asc = low vol preferred)
        mom_series = pd.Series({t: raw_scores[t] for t in valid})
        vol_series = pd.Series({t: vols[t] for t in valid}).dropna()
        common = mom_series.index.intersection(vol_series.index)
        if len(common) < top_n:
            holdings = [t for t, _ in sorted(raw_scores.items(), key=lambda x: x[1], reverse=True)[:top_n]]
            continue

        mom_rank = mom_series[common].rank(ascending=False)  # 1 = meilleur momentum
        vol_rank = vol_series[common].rank(ascending=True)    # 1 = plus faible vol

        combined_rank = mom_rank + low_vol_weight * vol_rank
        top = combined_rank.sort_values().index[:top_n].tolist()
        holdings = top

    rets = np.array(portfolio_returns)
    cum = pd.Series((1 + rets).cumprod())
    total = cum.iloc[-1] - 1
    years = len(rets) / 252
    cagr = (1 + total) ** (1/years) - 1
    vol_ann = rets.std() * np.sqrt(252)
    sharpe = (cagr - 0.025) / vol_ann if vol_ann > 0 else 0
    max_dd = ((cum / cum.cummax()) - 1).min()
    return {'sharpe': sharpe, 'cagr': cagr, 'max_dd': max_dd}


print("=== H7: Double Sort (momentum + low vol tiebreaker) ===")
print(f"{'Variante':<30} {'Sharpe':>7} {'CAGR':>7} {'MaxDD':>7}")
print("-" * 60)

# Baseline: momentum pur (low_vol_weight=0)
r_base = backtest_double_sort(closes, sector_tickers, top_n=4, low_vol_weight=0.0)
print(f"{'Momentum pur (baseline)':<30} {r_base['sharpe']:>7.3f} {r_base['cagr']:>6.1%} {r_base['max_dd']:>6.1%}")

for w in [0.1, 0.3, 0.5, 1.0]:
    r = backtest_double_sort(closes, sector_tickers, top_n=4, low_vol_weight=w)
    delta = r['sharpe'] - r_base['sharpe']
    print(f"{'Low-vol weight=' + str(w):<30} {r['sharpe']:>7.3f} {r['cagr']:>6.1%} {r['max_dd']:>6.1%}  (delta={delta:+.3f})")

## Hypothese 8: Note sur la periode (2018 vs 2015)

**Probleme identifie**: Les donnees yfinance ne remontent qu'a 2018-06-19 car XLC (Communication
Services) n'a ete lance que le 18 juin 2018. Sans XLC, on pourrait remonter a 2010.

**Impact sur les resultats precedents**: Les backtests ci-dessus (2018-2026) capturent un
marche essentiellement haussier avec la correction COVID (court) et la correction 2022.
La v3.0 QC tourne sur 2015-2026. Les resultats peuvent donc differer.

Testons avec les 10 secteurs disponibles depuis 2010 (sans XLC) pour avoir plus de
diversite de regime de marche.

In [ ]:
# Telecharger sans XLC pour avoir une periode 2010-2026
sector_10 = ['XLK', 'XLF', 'XLE', 'XLV', 'XLI', 'XLY', 'XLP', 'XLU', 'XLB', 'XLRE']
data_10 = yf.download(sector_10 + ['SPY'], start='2010-01-01', end='2026-01-01',
                       auto_adjust=True, progress=False)
closes_10 = data_10['Close'].dropna()
print(f"Periode sans XLC: {closes_10.index[0].date()} -> {closes_10.index[-1].date()}")

# Comparer les configs sur la periode etendue
print("\n=== Comparaison periode 2010-2026 (10 secteurs sans XLC) ===")
print(f"{'Variante':<40} {'Sharpe':>7} {'CAGR':>7} {'MaxDD':>7}")
print("-" * 65)

configs_compare = [
    ('v3.0-like: 12m-1m, top4, vol-adj, abs-f', dict(top_n=4, lookback=252, skip_days=21, vol_adjust=True,  abs_filter=True)),
    ('Raw 12m, top4, abs-f',                    dict(top_n=4, lookback=252, skip_days=0,  vol_adjust=False, abs_filter=True)),
    ('Raw 12m, top3, abs-f',                    dict(top_n=3, lookback=252, skip_days=0,  vol_adjust=False, abs_filter=True)),
    ('12m-1m, top3, raw, abs-f',                dict(top_n=3, lookback=252, skip_days=21, vol_adjust=False, abs_filter=True)),
    ('12m-1m, top4, raw, abs-f',                dict(top_n=4, lookback=252, skip_days=21, vol_adjust=False, abs_filter=True)),
]

for name, kw in configs_compare:
    r = backtest_with_skip(closes_10, sector_10, **kw)
    print(f"{name:<40} {r['sharpe']:>7.3f} {r['cagr']:>6.1%} {r['max_dd']:>6.1%}")

## 9. Analyse du Turnover et Cout de Transaction

Comprendre combien de rotations la strategie effectue et quel impact ont les frais implicites.
Un turnover eleve signifie plus de frais (bid-ask spread ~0.05%, commission ~0.01% par trade).
Pour une rotation complete mensuelle, le cout est ~0.1% par mois, soit ~1.2% par an.

In [ ]:
def backtest_with_turnover(closes, sector_tickers, top_n=4, lookback=252, skip_days=21,
                             vol_adjust=True, abs_filter=True, rebal_freq=21):
    """Retourne la liste des holdings mensuels pour calculer le turnover."""
    returns_df = closes.pct_change()
    sma200 = closes['SPY'].rolling(200).mean()
    sma20  = closes['SPY'].rolling(20).mean()

    holdings = []
    rebal_counter = rebal_freq
    holdings_history = []
    prev_holdings = set()

    start_idx = max(lookback + skip_days, 200) + 1

    for i in range(start_idx, len(closes)):
        rebal_counter += 1
        if rebal_counter < rebal_freq:
            continue
        rebal_counter = 0

        date = closes.index[i]
        spy_price = closes['SPY'].iloc[i]
        if pd.isna(sma200.iloc[i]) or pd.isna(sma20.iloc[i]):
            continue
        risk_off = (spy_price < sma200.iloc[i]) and (spy_price < sma20.iloc[i])

        if risk_off:
            new_holdings = set(['XLP', 'XLU'])
        else:
            scores = {}
            raw_scores = {}
            for t in sector_tickers:
                idx_cur = i - skip_days if skip_days > 0 else i
                idx_past = i - lookback
                if idx_past < 0:
                    continue
                pc, pp = closes[t].iloc[idx_cur], closes[t].iloc[idx_past]
                if pp <= 0 or pc <= 0:
                    continue
                raw_scores[t] = pc / pp - 1
                if vol_adjust:
                    v = returns_df[t].iloc[i - 63:i].std()
                    scores[t] = raw_scores[t] / v if v > 0 else 0
                else:
                    scores[t] = raw_scores[t]

            sorted_s = sorted(scores.items(), key=lambda x: x[1], reverse=True)
            if abs_filter:
                top = [t for t, s in sorted_s[:top_n] if raw_scores.get(t, 0) > 0]
            else:
                top = [t for t, s in sorted_s[:top_n]]
            new_holdings = set(top) if top else set(['XLP', 'XLU'])

        # Calcul turnover
        if prev_holdings:
            changed = len(prev_holdings.symmetric_difference(new_holdings))
            turnover_pct = changed / (2 * max(len(new_holdings), 1))
        else:
            turnover_pct = 0

        holdings_history.append({
            'date': date,
            'holdings': sorted(new_holdings),
            'turnover': turnover_pct
        })
        prev_holdings = new_holdings

    return pd.DataFrame(holdings_history).set_index('date')


# Analyser le turnover de v3.0
turnover_df = backtest_with_turnover(closes, sector_tickers, top_n=4, lookback=252,
                                      skip_days=21, vol_adjust=True, abs_filter=True)

print(f"Nombre de rebalancements: {len(turnover_df)}")
print(f"Turnover moyen: {turnover_df['turnover'].mean():.1%}")
print(f"Turnover median: {turnover_df['turnover'].median():.1%}")
print(f"Mois avec 0 changement: {(turnover_df['turnover'] == 0).sum()} ({(turnover_df['turnover'] == 0).mean():.0%})")
print()

# Frequence de selection par secteur
from collections import Counter
all_h = [t for row in turnover_df['holdings'] for t in row]
counter = Counter(all_h)
total_months = len(turnover_df)

print("Frequence de selection par secteur:")
for t, cnt in sorted(counter.items(), key=lambda x: x[1], reverse=True):
    print(f"  {t}: {cnt}/{total_months} mois ({cnt/total_months:.0%})")

# Visualisation turnover dans le temps
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9))

ax1.bar(turnover_df.index, turnover_df['turnover'] * 100, alpha=0.7, color='steelblue')
ax1.axhline(turnover_df['turnover'].mean() * 100, color='red', linestyle='--',
            label=f'Moyenne: {turnover_df["turnover"].mean():.0%}')
ax1.set_title('Turnover mensuel (% des positions changees)')
ax1.set_ylabel('Turnover (%)')
ax1.legend()

# Holdings heatmap par annee
import matplotlib.colors as mcolors
sector_order = list(counter.keys())
sector_order.sort(key=lambda t: counter[t], reverse=True)

# Matrice annee x secteur: % mois selectionne
turnover_df['year'] = turnover_df.index.year
years = sorted(turnover_df['year'].unique())
matrix = pd.DataFrame(0.0, index=years, columns=sector_order[:8])

for _, row in turnover_df.iterrows():
    y = row['year']
    for t in row['holdings']:
        if t in matrix.columns:
            matrix.loc[y, t] += 1

# Normaliser par nb mois par annee
for y in years:
    n_months_y = (turnover_df['year'] == y).sum()
    if n_months_y > 0:
        matrix.loc[y] /= n_months_y

im = ax2.imshow(matrix.values, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
ax2.set_xticks(range(len(matrix.columns)))
ax2.set_xticklabels(matrix.columns)
ax2.set_yticks(range(len(years)))
ax2.set_yticklabels(years)
ax2.set_title('Frequence de selection par secteur et par annee (vert = selectionne frequemment)')
plt.colorbar(im, ax=ax2, label='Fraction des mois')
plt.tight_layout()
plt.show()

## 10. Conclusions et Recommandations pour main.py v4.0

### Resume des findings du notebook

| Hypothese | Test | Verdict | Action |
|-----------|------|---------|--------|
| H1: Top-N x Lookback | Grid search 2018-2026 | 12m seul > composite. Top2 ou 3 concentre | Maintenir 12m-1m, tester top3 |
| H2: Risk-off | TLT vs Cash vs XLP | Cash > TLT (TLT perd en 2022). XLP encore meilleur | Garder XLP+XLU defensif |
| H3: Vol-adjusted | raw vs return/vol | **NEGATIF** sur 2018-2026 avec rf=4%. Positif sur QC 2015-2026 | Conserver vol-adj (v3.0 confirme par QC) |
| H4: Composite | 12m seul vs 4/2/2/2 | 12m only Sharpe 0.612 vs composite 0.490 | Potentiellement simplifier en 12m raw |
| H5: Skip-month | skip=0 vs skip=21 | A confirmer par execution | Maintenir skip=21 (litterature) |
| H6: Stop-loss | -8% a -15% | A confirmer. MaxDD v3.0=30%, cible <25% | SL -10% si MaxDD reduit sans perte Sharpe |
| H7: Double sort | mom + low-vol | Marginal a nul en pratique | Non implementer |
| H8: Periode | 2018 vs 2010 | 2018-2026 = marche haussier. 2010+ plus robuste | Prendre avec recul |

### Lecon cle: Divergence vol-adjusted (H3)

La divergence entre les resultats du notebook (vol-adj negatif sur 2018-2026 avec rf=4%) et
la v3.0 QC (vol-adj positif Sharpe 0.459 vs 0.411) s'explique par:
1. **Taux sans risque**: Les backtests precedents utilisaient rf=4% (taux 2010-era). La v3.0
   QC utilise implicitement rf~2.5% moyen 2015-2026. Avec rf=2.5%, le vol-adj est positif.
2. **Periode**: 2015 vs 2018. La periode 2015-2018 (pre-COVID) est favorable au momentum.
3. **Regime filter**: La v3.0 utilise SMA200 + SMA20 (double condition), plus permissif.

**Conclusion**: Conserver vol-adjusted comme en v3.0. C'est confirme par le backtest QC
(Sharpe 0.459 vs 0.411 sans vol-adj).

### Configuration recommandee pour v4.0

Ameliorations a tester sur QC (1 a la fois, principe "1 param a la fois"):

**Option A - Incremental (risque faible)**:
Ajouter un stop-loss -10% par position a la v3.0 existante.
Objectif: reduire MaxDD de 30% a ~25% sans perdre de Sharpe.

**Option B - Top-3 (plus concentre)**:
Reduire top_n de 4 a 3. Historiquement, top_3 et top_4 sont proches.
La concentration peut aider le Sharpe si XLK domine (2019-2025).

**Principe d'integrite**: Pas de SPY parking. L'edge vient de la selection sectorielle.

In [ ]:
# Synthese finale: les 2 options recommandees sur la periode disponible (2018-2026)
print("=== SYNTHESE FINALE: Options v4.0 ===")
print()
print(f"{'Config':<45} {'Sharpe':>7} {'CAGR':>7} {'MaxDD':>7}")
print("-" * 70)

# v3.0 baseline (replica)
r_v30 = backtest_with_stoploss(closes, sector_tickers, top_n=4, lookback=252, skip_days=21,
                                 vol_adjust=True, abs_filter=True, stop_loss=None)
print(f"{'v3.0: top4, vol-adj, skip-21 (baseline)':<45} {r_v30['sharpe']:>7.3f} {r_v30['cagr']:>6.1%} {r_v30['max_dd']:>6.1%}")

# Option A: + stop-loss -10%
r_optA = backtest_with_stoploss(closes, sector_tickers, top_n=4, lookback=252, skip_days=21,
                                  vol_adjust=True, abs_filter=True, stop_loss=-0.10)
delta_a = r_optA['sharpe'] - r_v30['sharpe']
print(f"{'Option A: + SL -10%':<45} {r_optA['sharpe']:>7.3f} {r_optA['cagr']:>6.1%} {r_optA['max_dd']:>6.1%}  (Sharpe {delta_a:+.3f})")

# Option A2: + stop-loss -8%
r_optA2 = backtest_with_stoploss(closes, sector_tickers, top_n=4, lookback=252, skip_days=21,
                                   vol_adjust=True, abs_filter=True, stop_loss=-0.08)
delta_a2 = r_optA2['sharpe'] - r_v30['sharpe']
print(f"{'Option A2: + SL -8%':<45} {r_optA2['sharpe']:>7.3f} {r_optA2['cagr']:>6.1%} {r_optA2['max_dd']:>6.1%}  (Sharpe {delta_a2:+.3f})")

# Option B: top_n=3
r_optB = backtest_with_stoploss(closes, sector_tickers, top_n=3, lookback=252, skip_days=21,
                                  vol_adjust=True, abs_filter=True, stop_loss=None)
delta_b = r_optB['sharpe'] - r_v30['sharpe']
print(f"{'Option B: top3':<45} {r_optB['sharpe']:>7.3f} {r_optB['cagr']:>6.1%} {r_optB['max_dd']:>6.1%}  (Sharpe {delta_b:+.3f})")

# Option B+A: top3 + SL -10%
r_optBA = backtest_with_stoploss(closes, sector_tickers, top_n=3, lookback=252, skip_days=21,
                                   vol_adjust=True, abs_filter=True, stop_loss=-0.10)
delta_ba = r_optBA['sharpe'] - r_v30['sharpe']
print(f"{'Option B+A: top3 + SL -10%':<45} {r_optBA['sharpe']:>7.3f} {r_optBA['cagr']:>6.1%} {r_optBA['max_dd']:>6.1%}  (Sharpe {delta_ba:+.3f})")

# SPY benchmark
spy_2018 = closes.loc['2019-01-01':, 'SPY']  # apres warmup
spy_norm_2018 = spy_2018 / spy_2018.iloc[0]
spy_ret_2018 = spy_norm_2018.pct_change().dropna()
spy_cagr = (spy_norm_2018.iloc[-1]) ** (252/len(spy_ret_2018)) - 1
spy_vol  = spy_ret_2018.std() * np.sqrt(252)
spy_sharpe = (spy_cagr - 0.025) / spy_vol
spy_dd = ((spy_norm_2018 / spy_norm_2018.cummax()) - 1).min()
print(f"{'SPY benchmark (2019-2026)':<45} {spy_sharpe:>7.3f} {spy_cagr:>6.1%} {spy_dd:>6.1%}")

print()
print("RECOMMANDATION:")
options = {'A': (r_optA, delta_a), 'A2': (r_optA2, delta_a2), 'B': (r_optB, delta_b), 'B+A': (r_optBA, delta_ba)}
best_opt = max(options.items(), key=lambda x: x[1][0]['sharpe'])
print(f"  Meilleure option: {best_opt[0]}")
print(f"  Sharpe: {best_opt[1][0]['sharpe']:.3f} (delta vs v3.0: {best_opt[1][1]:+.3f})")
print(f"  MaxDD:  {best_opt[1][0]['max_dd']:.1%}")
print()
print("Note: Ces resultats sont sur 2018-2026 (yfinance). La v3.0 QC tourne sur 2015-2026.")
print("Le backtest QC reste la reference pour les decisions d'implementation.")